# Qwen3.5-9B · 추론 해상도 5단계 비교

**v1.1: Windows CRLF/LF 코드 해시 비교 수정. 기존 baseline·체크포인트 수정 없이 사용합니다.**

**384² → 448² → 560² → 672² → 784²**, 동일 LoRA·동일 검증 약 500개로 직렬 실행합니다.
학습 해상도는 기존 384²를 유지하며 재학습하지 않습니다. 최종 holdout·test·dev는 평가하지 않습니다.

### 실행 방법
1. `TASK006_Baseline_Qwen35_9B.ipynb`의 audit → train → lora_eval이 정상 완료되어 있어야 합니다.
2. 이 노트북을 **baseline을 실행한 같은 프로젝트 폴더**에서 열고 아래 설정을 확인하세요.
3. `BASELINE_RUN_DIR`에 baseline 실행 때 출력된 `output/TASK-006/TASK006-...` 폴더를 지정합니다. 후보가 하나면 자동 선택됩니다.
4. **Run All**하면 준비 → 해상도 5개 → 최종 표 순서로 실행됩니다. 모델 설치·재학습·전체 test 추론은 없습니다.

각 조건 완료 시 화면에 표를 표시하고 `resolution_comparison.csv`를 갱신합니다.
완료 조건은 재실행 시 건너뜁니다. 중단된 조건은 부분 로그를 보존하고 그 해상도의 첫 문항부터 다시 평가합니다.

원본 baseline에서 파생된 **사용자의 저장 실행 코드**를 해시 검증 후 불러옵니다. 입력 처리·프롬프트·파싱·NF4 설정을 그대로 사용하고 픽셀 예산만 바꿉니다.
픽셀 예산은 정사각형 강제 resize가 아닙니다. 종횡비와 processor 정렬에 따라 실제 크기·시각 토큰 수가 달라지며 문항별로 기록합니다.

384²도 이번 노트북에서 다시 측정합니다. 모든 조건에 워밍업 1문항을 적용하고 모델 로딩/워밍업 시간은 문항당 추론 시간에서 제외합니다.
기존 384² 예측과 재현 여부를 함께 저장합니다. 미세한 점수 차이만으로 채택하지 않습니다.


## 1. 기준 실행 폴더 지정
다른 컴퓨터에서 파일을 복사할 필요는 없습니다. 해당 컴퓨터에서 완료한 baseline 출력 폴더를 사용합니다.

In [1]:
from pathlib import Path
import os, sys, json, hashlib, subprocess, time
PROJECT_DIR = Path.cwd().resolve()
BASELINE_RUN_DIR = None  # 예: PROJECT_DIR / "output/TASK-006/TASK006-xxxxxxxxxxxxxxxx"
ENV_PYTHON = PROJECT_DIR / "downloads/envs/TASK006_baseline_qwen35" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
RESOLUTIONS = [384, 448, 560, 672, 784]
SESSION_TAG = "inference_resolution_5_v1"
RETRY_OOM = False  # 이미 OOM으로 기록된 조건도 재시도하려면 True

if BASELINE_RUN_DIR is None:
    candidates = sorted(p.parent for p in (PROJECT_DIR / "output/TASK-006").glob("TASK006-*/lora_eval_complete.json"))
    if len(candidates) != 1:
        print("baseline 후보:")
        for p in candidates: print(p)
        raise RuntimeError("BASELINE_RUN_DIR에 비교할 baseline 결과 폴더를 지정하세요.")
    BASELINE_RUN_DIR = candidates[0]
BASELINE_RUN_DIR = Path(BASELINE_RUN_DIR).resolve()
for name in ["run_config.json","task006_worker.py","audit_complete.json","train_complete.json","lora_eval_complete.json","requirements.lock.txt","model_assets.json"]:
    if not (BASELINE_RUN_DIR / name).is_file(): raise FileNotFoundError(BASELINE_RUN_DIR / name)
if not ENV_PYTHON.is_file(): raise FileNotFoundError(f"baseline 전용 Python 경로를 확인하세요: {ENV_PYTHON}")
if len(RESOLUTIONS)!=5 or RESOLUTIONS[0]!=384 or len(set(RESOLUTIONS))!=5 or any(not isinstance(x,int) or x<=0 for x in RESOLUTIONS):
    raise ValueError("384를 첫 값으로 하는 서로 다른 양의 정수 해상도 5개가 필요합니다.")
print("baseline:",BASELINE_RUN_DIR)
print("순차 비교:",RESOLUTIONS)


baseline: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006\TASK006-8f2c71505c11284c
순차 비교: [384, 448, 560, 672, 784]


## 2. 실행 함수 정의
아래 코드는 별도 GPU 프로세스로 실행됩니다. 기존 checkpoint와 분할을 수정하지 않습니다.

In [2]:
RESOLUTION_WORKER = r'''
import csv, hashlib, importlib.util, json, os, sys, time, uuid
from pathlib import Path


def digest(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()


def verify_worker_source(path, expected):
    data=Path(path).read_bytes()
    raw=hashlib.sha256(data).hexdigest()
    lf=hashlib.sha256(data.replace(b"\r\n",b"\n")).hexdigest()
    if expected not in (raw,lf):
        raise RuntimeError(f"baseline 코드 내용이 다릅니다. expected={expected}, raw={raw}, LF={lf}. 기존 파일/설정을 수정하지 말고 실행 폴더를 확인하세요.")
    return expected


def read(path): return json.loads(Path(path).read_text(encoding='utf-8'))


def write(path,obj):
    p=Path(path);tmp=p.with_name(p.name+'.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding='utf-8');tmp.replace(p)


def verified_status(root,size):
    p=root/f'res_{size}_status.json'
    if not p.exists(): return {'state':'not_run'}
    rec=read(p)
    if rec['state']=='completed':
        for name,sha in rec['artifacts'].items():
            f=Path(rec['directory'])/name
            if not f.is_file() or digest(f)!=sha: raise RuntimeError(f'완료 결과 변경/누락: {f}')
    return rec


def paired(a,b):
    m=a.merge(b,on='id',validate='one_to_one',suffixes=('_384','_new'))
    if len(m)!=len(a) or len(m)!=len(b) or not (m.gold_384==m.gold_new).all():
        raise RuntimeError('비교 ID/정답이 동일하지 않습니다.')
    if not (m.group_id_384==m.group_id_new).all(): raise RuntimeError('이미지 그룹 불일치')
    old=m.answer_384==m.gold_384;new=m.answer_new==m.gold_new
    m['transition']=['gain' if y and not x else 'loss' if x and not y else 'same_correct' if x else 'same_wrong' for x,y in zip(old,new)]
    return m,{'gain':int((~old & new).sum()),'loss':int((old & ~new).sum()),
              'delta_pp':100*float(new.mean()-old.mean())}


def summarize(cfg):
    import pandas as pd
    root=Path(cfg['output_dir']);rows=[];ref=None
    r=verified_status(root,384)
    if r['state']=='completed':ref=pd.read_csv(Path(r['directory'])/'valid_predictions.csv',keep_default_na=False)
    types=[]
    for size in cfg['resolutions']:
        rec=verified_status(root,size);row={'resolution':size,'pixel_budget':size*size,'status':rec['state']}
        if rec['state']=='completed':
            d=Path(rec['directory']);metrics=read(d/'valid_metrics.json');row.update(metrics)
            row['accuracy_pct']=100*metrics['accuracy']
            row['parse_failure_pct']=100*metrics['parse_failure_rate']
            pred=pd.read_csv(d/'valid_predictions.csv',keep_default_na=False)
            if ref is not None:
                joined,stats=paired(ref,pred);row.update(stats)
                joined.to_csv(root/f'paired_384_vs_{size}.csv',index=False,encoding='utf-8-sig')
                joined[joined.transition.isin(['gain','loss'])].to_csv(root/f'changed_384_vs_{size}.csv',index=False,encoding='utf-8-sig')
            for name,g in pred.groupby('question_type'):
                types.append({'resolution':size,'question_type':name,'n':len(g),'correct_n':int((g.answer==g.gold).sum()),'accuracy':float((g.answer==g.gold).mean())})
            row['result_directory']=str(d)
        else:row['error']=rec.get('error','')
        rows.append(row)
    table=pd.DataFrame(rows);table.to_csv(root/'resolution_comparison.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(types,columns=['resolution','question_type','n','correct_n','accuracy']).to_csv(root/'type_comparison.csv',index=False,encoding='utf-8-sig')
    complete=[x['resolution'] for x in rows if x['status']=='completed']
    write(root/'summary.json',{'completed':complete,'total_conditions':len(rows),'adoption':'not_selected',
        'baseline_run':cfg['baseline_dir'],'test_used':False,'holdout_evaluated':False,'review_02':'pending','results':rows})
    (root/'PROJECT_STATUS_update.md').write_text('# TASK-006 추론 해상도 비교\n\n완료 조건: '+str(complete)+'\n\n동일 저장 LoRA/고정 검증셋. 재학습·test·holdout 평가 없음. 채택/독립 검토 미완료.\n\n결과: '+str(root/'resolution_comparison.csv'),encoding='utf-8')
    print(table.to_string(index=False),flush=True)
    return rows


def load_baseline(cfg):
    base=Path(cfg['baseline_dir']);original=read(base/'run_config.json')
    worker=base/'task006_worker.py'
    if verify_worker_source(worker,original['worker_sha256'])!=cfg['baseline_worker_sha256']:
        raise RuntimeError('baseline 실행 코드 해시 불일치')
    spec=importlib.util.spec_from_file_location('saved_baseline',worker)
    module=importlib.util.module_from_spec(spec);sys.modules[spec.name]=module;spec.loader.exec_module(module)
    module.block_network()
    return module,original


def prepare(cfg):
    import pandas as pd
    root=Path(cfg['output_dir']);base=Path(cfg['baseline_dir']);module,original=load_baseline(cfg)
    train=module.completed(base,'train');evaluation=module.completed(base,'lora_eval')
    audit=module.completed(base,'audit')
    if train is None or evaluation is None or audit is None:raise RuntimeError('baseline audit/train/lora_eval을 먼저 완료하세요.')
    if original['pixel_budget']!=384**2:raise RuntimeError('학습 해상도 384²인 baseline을 지정하세요.')
    reload_check=read(Path(train['directory'])/'reload_check.json')
    if not reload_check.get('identical_outputs'):raise RuntimeError('baseline 저장/재로드 검증 미통과')
    _,valid,info=module.load_split(original,base)
    # Avoid relying only on paths; check every model asset against baseline download hashes.
    assets=read(base/'model_assets.json')
    if assets['revision']!=original['revision']:raise RuntimeError('모델 revision 불일치')
    for name,record in assets['files'].items():
        f=Path(original['model_dir'])/name
        if not f.is_file() or digest(f)!=record['sha256']:raise RuntimeError(f'모델 파일 확인 필요: {name}')
    valid[['id','group_id','question_type']].to_csv(root/'fixed_valid_ids.csv',index=False)
    frozen={'baseline_config':original,'data_manifest':info,'checkpoint':str(Path(train['directory'])/'adapter_epoch1'),
            'baseline_records':{name:digest(base/(name+'_complete.json')) for name in ['audit','train','lora_eval']},
            'baseline_config_sha256':digest(base/'run_config.json'),'valid_ids_sha256':digest(root/'fixed_valid_ids.csv'),
            'valid_n':len(valid),'previous_predictions':str(Path(evaluation['directory'])/'valid_lora_predictions.csv'),
            'model_file_stats':{name:[(Path(original['model_dir'])/name).stat().st_size,(Path(original['model_dir'])/name).stat().st_mtime_ns] for name in assets['files']}}
    frozen_path=root/'frozen_inputs.json'
    if frozen_path.exists() and read(frozen_path)!=frozen:raise RuntimeError('기준 입력 변경. 새 SESSION_TAG로 실행하세요.')
    write(frozen_path,frozen)
    print('준비 완료. 저장 LoRA:',frozen['checkpoint'],'검증:',len(valid),flush=True)
    summarize(cfg)


def run_resolution(cfg,size):
    import torch,pandas as pd
    from peft import PeftModel
    root=Path(cfg['output_dir']);base=Path(cfg['baseline_dir']);frozen=read(root/'frozen_inputs.json')
    module,original=load_baseline(cfg)
    if digest(base/'run_config.json')!=frozen['baseline_config_sha256']:raise RuntimeError('baseline 설정 변경')
    for name,sha in frozen['baseline_records'].items():
        if digest(base/(name+'_complete.json'))!=sha:raise RuntimeError('baseline 완료 기록 변경')
    # Revalidate checkpoint, split and current image bytes before reuse/evaluation.
    module.completed(base,'train')
    _,valid,info=module.load_split(original,base)
    if digest(root/'fixed_valid_ids.csv')!=frozen['valid_ids_sha256']:raise RuntimeError('고정 검증 ID 파일 변경')
    ids=pd.read_csv(root/'fixed_valid_ids.csv',dtype=str)
    if ids.id.tolist()!=valid.id.tolist():raise RuntimeError('검증 ID 순서 변경')
    for name,stat in frozen['model_file_stats'].items():
        f=Path(original['model_dir'])/name
        if [f.stat().st_size,f.stat().st_mtime_ns]!=stat:raise RuntimeError('모델 파일 변경: 준비 셀 재실행 필요')
    previous=verified_status(root,size)
    if previous['state']=='completed' or (previous['state']=='blocked_oom' and not cfg['retry_oom']):
        print('기존 상태 재사용:',size,previous['state'],flush=True);summarize(cfg);return
    out=root/f'res_{size}_{time.strftime("%Y%m%d_%H%M%S")}_{uuid.uuid4().hex[:8]}';out.mkdir()
    settings=dict(original);settings['pixel_budget']=size*size;settings['image_policy']=f'inference pixel budget {size}²; original aspect ratio'
    settings['run_dir']=str(out)
    write(out/'inference_config.json',settings)
    status_path=root/f'res_{size}_status.json'
    write(status_path,{'state':'running','directory':str(out)})
    try:
        module.gpu_environment(settings,out)
        adapter=module.ModelAdapter(settings,out)
        adapter.model=PeftModel.from_pretrained(adapter.model,frozen['checkpoint'],local_files_only=True,is_trainable=False)
        adapter.model.eval()
        # No adapter.add_lora / optimizer / training. Warm up same first validation input.
        probe=valid.iloc[0].to_dict();probe.pop('answer',None)
        adapter.generate(adapter.encode(probe,training=False));torch.cuda.synchronize()
        _,metrics=module.evaluate(adapter,valid,'valid',out)
        metrics.update({'checkpoint':frozen['checkpoint'],'train_pixel_budget':original['pixel_budget'],
                        'inference_pixel_budget':size*size,'warmup_samples':1,'timing':'validation loop only, excludes load/warmup'})
        write(out/'valid_metrics.json',metrics)
        if size==384:
            old=pd.read_csv(frozen['previous_predictions'],keep_default_na=False)
            new=pd.read_csv(out/'valid_predictions.csv',keep_default_na=False)
            m,_=paired(old,new)
            write(out/'baseline_reproduction.json',{'n':len(m),'same_predictions':bool((m.answer_384==m.answer_new).all()),
                'same_raw_outputs':bool((m.raw_output_384==m.raw_output_new).all()),
                'changed_prediction_count':int((m.answer_384!=m.answer_new).sum()),
                'note':'timing rerun with warmup; discrepancy requires review'})
        artifacts={str(p.relative_to(out)):digest(p) for p in out.rglob('*') if p.is_file()}
        write(status_path,{'state':'completed','directory':str(out),'artifacts':artifacts})
    except torch.cuda.OutOfMemoryError as exc:
        # No smaller resolution, changed quantization, or partial accuracy substituted.
        write(status_path,{'state':'blocked_oom','directory':str(out),'error':str(exc),
            'partial_predictions_are_not_metrics':True})
        print('OOM으로 해당 조건 차단:',size,flush=True)
    except BaseException as exc:
        write(status_path,{'state':'failed_or_interrupted','directory':str(out),'error':str(exc)})
        summarize(cfg);raise
    finally:
        with open(root/'CHANGELOG.md','a',encoding='utf-8') as f:
            f.write(f"\n- {time.strftime('%Y-%m-%d %H:%M:%S')} / {size}² / {read(status_path)['state']} / {out}\n")
    summarize(cfg)


if __name__=='__main__':
    config=read(sys.argv[1]);stage=sys.argv[2]
    if stage=='prepare':prepare(config)
    elif stage=='summary':summarize(config)
    else:
        size=int(stage)
        if size not in config['resolutions']:raise ValueError('설정에 없는 해상도')
        run_resolution(config,size)

'''

In [3]:
def file_hash(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(4*1024*1024),b""): h.update(b)
    return h.hexdigest()
base_config=json.loads((BASELINE_RUN_DIR/"run_config.json").read_text(encoding="utf-8"))
if base_config.get("model_id")!="Qwen/Qwen3.5-9B":raise RuntimeError("Qwen3.5-9B baseline을 지정하세요.")
worker_bytes=(BASELINE_RUN_DIR/"task006_worker.py").read_bytes()
worker_raw_hash=hashlib.sha256(worker_bytes).hexdigest()
worker_lf_hash=hashlib.sha256(worker_bytes.replace(bytes([13,10]),bytes([10]))).hexdigest()
if base_config["worker_sha256"] not in (worker_raw_hash,worker_lf_hash):
    raise RuntimeError(f"baseline 코드 내용이 다릅니다. expected={base_config['worker_sha256']}, raw={worker_raw_hash}, LF={worker_lf_hash}. 기존 파일/설정을 수정하지 말고 실행 폴더를 확인하세요.")
if worker_raw_hash!=base_config["worker_sha256"]:
    print("Windows CRLF 줄바꿈 차이만 확인됨. LF 정규화 해시 검증 통과.")
# Detect package changes instead of silently comparing different environments.
freeze=subprocess.check_output([str(ENV_PYTHON),"-m","pip","freeze"],text=True,encoding="utf-8")
old=(BASELINE_RUN_DIR/"requirements.lock.txt").read_text(encoding="utf-8")
if sorted(freeze.splitlines())!=sorted(old.splitlines()):
    raise RuntimeError("baseline 실행 후 패키지 구성이 바뀌었습니다. baseline 전용 환경을 복원한 뒤 실행하세요.")
CFG={"baseline_dir":str(BASELINE_RUN_DIR),"resolutions":RESOLUTIONS,"session_tag":SESSION_TAG,
     "baseline_worker_sha256":base_config["worker_sha256"],
     "baseline_config_sha256":file_hash(BASELINE_RUN_DIR/"run_config.json"),
     "train_record_sha256":file_hash(BASELINE_RUN_DIR/"train_complete.json"),
     "audit_record_sha256":file_hash(BASELINE_RUN_DIR/"audit_complete.json"),
     "worker_sha256":hashlib.sha256(RESOLUTION_WORKER.encode()).hexdigest(),
     "environment_sha256":hashlib.sha256(freeze.encode()).hexdigest(),"code_version":"resolution-5-v1.1-newline-fix"}
fingerprint=hashlib.sha256(json.dumps(CFG,sort_keys=True).encode()).hexdigest()[:16]
OUTPUT_DIR=PROJECT_DIR/"output/TASK-006-resolution"/("RES5-"+fingerprint)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
CFG.update(output_dir=str(OUTPUT_DIR),retry_oom=RETRY_OOM)
WORKER=OUTPUT_DIR/"resolution_worker.py";CONFIG=OUTPUT_DIR/"resolution_config.json"
compile(RESOLUTION_WORKER,str(WORKER),"exec")
WORKER.write_bytes(RESOLUTION_WORKER.encode("utf-8"))
CONFIG.write_text(json.dumps(CFG,ensure_ascii=False,indent=2),encoding="utf-8")
(OUTPUT_DIR/"requirements.lock.txt").write_text(freeze,encoding="utf-8")

def show_table():
    import csv
    from IPython.display import display,Markdown,FileLink
    p=OUTPUT_DIR/"resolution_comparison.csv"
    if not p.exists(): return
    with open(p,encoding="utf-8-sig",newline="") as f: rows=list(csv.DictReader(f))
    columns=[("resolution","해상도"),("status","상태"),("accuracy_pct","Accuracy(%)"),("correct_n","정답 수"),
             ("n","검증 수"),("delta_pp","384 대비 %p"),("gain","개선"),("loss","악화"),
             ("parse_failure_pct","파싱 실패(%)"),("seconds_per_sample","초/문항"),("peak_allocated_gib","최대 VRAM(GiB)")]
    def fmt(value):
        if value in (None,""): return "—"
        try: return f"{float(value):.3f}" if any(c in str(value) for c in '.eE') else str(value)
        except ValueError: return str(value).replace('|','/')
    text='| '+' | '.join(b for a,b in columns)+' |\n| '+' | '.join('---' for _ in columns)+' |\n'
    for row in rows: text+='| '+' | '.join(fmt(row.get(a,'')) for a,b in columns)+' |\n'
    display(Markdown(text));print("결과 파일:",p);display(FileLink(str(p)))

def run_stage(stage):
    # Same resolution project lock prevents simultaneous notebook runs.
    lock=PROJECT_DIR/"output/TASK-006-resolution/gpu_experiment.lock"
    if (BASELINE_RUN_DIR/"running.lock").exists():raise RuntimeError("baseline 작업이 실행 중입니다. 종료 후 실행하세요.")
    try: fd=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)
    except FileExistsError:raise RuntimeError(f"다른 해상도 실험이 실행 중이거나 잠금이 남아 있습니다: {lock}. 실행 프로세스가 없을 때만 잠금을 삭제하세요.")
    process=None
    try:
        with os.fdopen(fd,"w") as f:f.write(str(os.getpid()))
        env=os.environ.copy();env.update(PYTHONIOENCODING="utf-8",PYTHONUNBUFFERED="1")
        with open(OUTPUT_DIR/(str(stage)+".log"),"a",encoding="utf-8") as log:
            process=subprocess.Popen([str(ENV_PYTHON),"-u",str(WORKER),str(CONFIG),str(stage)],
                stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding="utf-8",errors="replace",env=env)
            for line in process.stdout:print(line,end="");log.write(line);log.flush()
            if process.wait()!=0:raise RuntimeError(f"단계 {stage} 실패. {OUTPUT_DIR / (str(stage)+'.log')} 확인")
    except BaseException:
        if process is not None and process.poll() is None:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
        if str(stage).isdigit():
            status=OUTPUT_DIR/("res_"+str(stage)+"_status.json")
            if status.exists():
                rec=json.loads(status.read_text(encoding="utf-8"))
                if rec["state"]=="running":
                    rec["state"]="interrupted";status.write_text(json.dumps(rec,ensure_ascii=False,indent=2),encoding="utf-8")
        raise
    finally:lock.unlink(missing_ok=True)
    show_table()
print("새 실험 결과 폴더:",OUTPUT_DIR)


Windows CRLF 줄바꿈 차이만 확인됨. LF 정규화 해시 검증 통과.
새 실험 결과 폴더: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-resolution\RES5-64d6ac7377998c2b


## 3. 기준 입력 확인
checkpoint·코드·패키지·모델 파일·분할·이미지 해시를 확인합니다. 모델 파일 전체를 확인하므로 이 단계는 시간이 걸릴 수 있습니다. 최종 검증은 평가하지 않습니다.

In [4]:
run_stage("prepare")

준비 완료. 저장 LoRA: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006\TASK006-8f2c71505c11284c\train_20260922_115654_f6f6586f\adapter_epoch1 검증: 500
 resolution  pixel_budget  status error
        384        147456 not_run      
        448        200704 not_run      
        560        313600 not_run      
        672        451584 not_run      
        784        614656 not_run      


| 해상도 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 384 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 384 | not_run | — | — | — | — | — | — | — | — | — |
| 448 | not_run | — | — | — | — | — | — | — | — | — |
| 560 | not_run | — | — | — | — | — | — | — | — | — |
| 672 | not_run | — | — | — | — | — | — | — | — | — |
| 784 | not_run | — | — | — | — | — | — | — | — | — |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-resolution\RES5-64d6ac7377998c2b\resolution_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-resolution\RES5-64d6ac7377998c2b\resolution_comparison.csv

## 4. 조건 1/5 — 384²
설정 셀 `RESOLUTIONS[0]`를 사용합니다. 조건이 완료되면 아래에 누적 비교표가 표시됩니다.

In [5]:
run_stage(RESOLUTIONS[0])

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 13:29:36.148000 6328 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:37,  1.47it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: Fu

| 해상도 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 384 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 384 | completed | 83.400 | 417.000 | 500.000 | 0.000 | 0.000 | 0.000 | 0.000 | 0.576 | 7.602 |
| 448 | not_run | — | — | — | — | — | — | — | — | — |
| 560 | not_run | — | — | — | — | — | — | — | — | — |
| 672 | not_run | — | — | — | — | — | — | — | — | — |
| 784 | not_run | — | — | — | — | — | — | — | — | — |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-resolution\RES5-64d6ac7377998c2b\resolution_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-resolution\RES5-64d6ac7377998c2b\resolution_comparison.csv

## 5. 조건 2/5 — 448²
설정 셀 `RESOLUTIONS[1]`를 사용합니다. 조건이 완료되면 아래에 누적 비교표가 표시됩니다.

In [6]:
run_stage(RESOLUTIONS[1])

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 13:34:50.115000 19212 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:38,  1.46it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

| 해상도 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 384 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 384 | completed | 83.400 | 417.000 | 500.000 | 0.000 | 0.000 | 0.000 | 0.000 | 0.576 | 7.602 |
| 448 | completed | 87.200 | 436.000 | 500.000 | 3.800 | 31.000 | 12.000 | 0.000 | 0.606 | 7.612 |
| 560 | not_run | — | — | — | — | — | — | — | — | — |
| 672 | not_run | — | — | — | — | — | — | — | — | — |
| 784 | not_run | — | — | — | — | — | — | — | — | — |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-resolution\RES5-64d6ac7377998c2b\resolution_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-resolution\RES5-64d6ac7377998c2b\resolution_comparison.csv

## 6. 조건 3/5 — 560²
설정 셀 `RESOLUTIONS[2]`를 사용합니다. 조건이 완료되면 아래에 누적 비교표가 표시됩니다.

In [7]:
run_stage(RESOLUTIONS[2])

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 13:40:19.710000 25920 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:45,  1.44it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

| 해상도 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 384 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 384 | completed | 83.400 | 417.000 | 500.000 | 0.000 | 0.000 | 0.000 | 0.000 | 0.576 | 7.602 |
| 448 | completed | 87.200 | 436.000 | 500.000 | 3.800 | 31.000 | 12.000 | 0.000 | 0.606 | 7.612 |
| 560 | completed | 89.200 | 446.000 | 500.000 | 5.800 | 39.000 | 10.000 | 0.000 | 0.705 | 7.632 |
| 672 | not_run | — | — | — | — | — | — | — | — | — |
| 784 | not_run | — | — | — | — | — | — | — | — | — |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-resolution\RES5-64d6ac7377998c2b\resolution_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-resolution\RES5-64d6ac7377998c2b\resolution_comparison.csv

## 7. 조건 4/5 — 672²
설정 셀 `RESOLUTIONS[3]`를 사용합니다. 조건이 완료되면 아래에 누적 비교표가 표시됩니다.

In [8]:
run_stage(RESOLUTIONS[3])

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 13:46:39.060000 4428 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:40,  1.46it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: Fu

| 해상도 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 384 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 384 | completed | 83.400 | 417.000 | 500.000 | 0.000 | 0.000 | 0.000 | 0.000 | 0.576 | 7.602 |
| 448 | completed | 87.200 | 436.000 | 500.000 | 3.800 | 31.000 | 12.000 | 0.000 | 0.606 | 7.612 |
| 560 | completed | 89.200 | 446.000 | 500.000 | 5.800 | 39.000 | 10.000 | 0.000 | 0.705 | 7.632 |
| 672 | completed | 93.200 | 466.000 | 500.000 | 9.800 | 55.000 | 6.000 | 0.000 | 0.969 | 7.652 |
| 784 | not_run | — | — | — | — | — | — | — | — | — |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-resolution\RES5-64d6ac7377998c2b\resolution_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-resolution\RES5-64d6ac7377998c2b\resolution_comparison.csv

## 8. 조건 5/5 — 784²
설정 셀 `RESOLUTIONS[4]`를 사용합니다. 조건이 완료되면 아래에 누적 비교표가 표시됩니다.

In [9]:
run_stage(RESOLUTIONS[4])

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 13:55:13.632000 11444 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<10:17,  1.23it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

| 해상도 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 384 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 384 | completed | 83.400 | 417 | 500 | 0.000 | 0 | 0 | 0.000 | 0.576 | 7.602 |
| 448 | completed | 87.200 | 436 | 500 | 3.800 | 31 | 12 | 0.000 | 0.606 | 7.612 |
| 560 | completed | 89.200 | 446 | 500 | 5.800 | 39 | 10 | 0.000 | 0.705 | 7.632 |
| 672 | completed | 93.200 | 466 | 500 | 9.800 | 55 | 6 | 0.000 | 0.969 | 7.652 |
| 784 | completed | 93.200 | 466 | 500 | 9.800 | 56 | 7 | 0.000 | 1.209 | 7.702 |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-resolution\RES5-64d6ac7377998c2b\resolution_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-resolution\RES5-64d6ac7377998c2b\resolution_comparison.csv

## 9. 최종 비교표
OOM은 blocked_oom으로 표시하며 부분 예측을 정확도로 집계하지 않습니다. 다른 오류는 잘못된 조건 비교를 피하기 위해 중단합니다.

In [10]:
run_stage("summary")

 resolution  pixel_budget    status   n  correct_n  accuracy  parse_failure_rate  fallback_usage_rate    seconds  seconds_per_sample  peak_allocated_gib  peak_reserved_gib                                                                                                                  checkpoint  train_pixel_budget  inference_pixel_budget  warmup_samples                                     timing  accuracy_pct  parse_failure_pct  gain  loss  delta_pp                                                                                                       result_directory
        384        147456 completed 500        417     0.834                 0.0                  0.0 287.799316            0.575599            7.601601          11.333984 C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006\TASK006-8f2c71505c11284c\train_20260922_115654_f6f6586f\adapter_epoch1              147456                  147456               1 validation loop only, excludes load/warmup          83.4               

| 해상도 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 384 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 384 | completed | 83.400 | 417 | 500 | 0.000 | 0 | 0 | 0.000 | 0.576 | 7.602 |
| 448 | completed | 87.200 | 436 | 500 | 3.800 | 31 | 12 | 0.000 | 0.606 | 7.612 |
| 560 | completed | 89.200 | 446 | 500 | 5.800 | 39 | 10 | 0.000 | 0.705 | 7.632 |
| 672 | completed | 93.200 | 466 | 500 | 9.800 | 55 | 6 | 0.000 | 0.969 | 7.652 |
| 784 | completed | 93.200 | 466 | 500 | 9.800 | 56 | 7 | 0.000 | 1.209 | 7.702 |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-resolution\RES5-64d6ac7377998c2b\resolution_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-resolution\RES5-64d6ac7377998c2b\resolution_comparison.csv

## 결과 확인

**가장 먼저 `resolution_comparison.csv`를 확인하세요.** 같은 표가 각 실행 셀 아래에도 표시됩니다.

| 파일 | 내용 |
|---|---|
| resolution_comparison.csv | 5개 조건 상태, Accuracy·정답 수·384 대비 변화·시간·VRAM |
| res_해상도_실행시각/valid_predictions.csv | 원출력·예측·정답·파싱 실패·그룹·실제 이미지 크기/시각 토큰 |
| res_해상도_실행시각/valid_metrics.json | 조건별 지표 원본 |
| paired_384_vs_해상도.csv | 동일 문항별 정오 비교 |
| changed_384_vs_해상도.csv | 384 대비 개선·악화 문항만 |
| type_comparison.csv | 기존 임시 문항 유형별 비교 |
| res_384_실행시각/baseline_reproduction.json | 기존 baseline 384 예측과 재실행 예측 비교 |
| summary.json | 완료 조건 및 상태 요약 |
| PROJECT_STATUS_update.md / CHANGELOG.md | 반영용 기록. 공식 PROJECT_STATUS는 수정하지 않음 |

- 실제 검증 개수는 기존 분할을 그대로 따릅니다. 새 500개를 추출하지 않습니다.
- 정답 수가 늘어도 작은 차이만으로 채택하지 않습니다. 개선/악화 문항과 비용을 함께 확인하세요.
- 기존 baseline 384 예측이 재현되지 않으면 `baseline_reproduction.json`과 환경 로그부터 검토하세요.
- 학습 해상도 비교는 다음 실험입니다. 이 파일에는 optimizer·재학습·제출 기능이 없습니다.
- 작성 시 문법/CPU 로직을 검사했습니다. 작성 환경에서 실제 GPU 실행은 하지 않았습니다.
